# 02 - Pointwise Curve Loss Terms

This notebook isolates the elementwise curve-residual losses in
`LossComponents`: MSE, L1, Huber, and Charbonnier. We feed each a residual
`diff` that we sweep across a controlled range and plot the resulting loss
value. This confirms the expected shape of each penalty: quadratic vs linear
vs the hybrid transitions of Huber and Charbonnier.

Modules exercised: `pipelines.backbone_pipeline.loss.LossComponents`
(`mse_diff`, `l1_diff`, `huber_diff`, `charbonnier_diff`).

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path("../..").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy             as np
import torch
import matplotlib.pyplot as plt

SEED   = 0
DEVICE = torch.device("cpu")

np.random.seed(SEED)
torch.manual_seed(SEED)
torch.use_deterministic_algorithms(True, warn_only=True)

plt.rcParams.update({
    "figure.dpi"     : 120,
    "savefig.dpi"    : 300,
    "font.size"      : 10,
    "axes.grid"      : True,
    "grid.alpha"     : 0.3,
    "axes.spines.top"   : False,
    "axes.spines.right" : False,
})

print("repo root :", REPO_ROOT)
print("torch     :", torch.__version__)
print("device    :", DEVICE)


## Penalty as a function of a scalar residual

Each `*_diff` helper reduces a residual tensor to a scalar by averaging. For
a constant residual `d` the mean equals the elementwise penalty at `d`, so we
can read off the penalty curve directly by sweeping `d`.

In [ ]:
from pipelines.backbone_pipeline.loss import LossComponents as LC

deltas      = torch.linspace(-3.0, 3.0, 241)
huber_delta = 1.0
charb_eps   = 1e-3

mse_vals   = torch.tensor([LC.mse_diff(torch.full((64,), float(d)))                       for d in deltas])
l1_vals    = torch.tensor([LC.l1_diff(torch.full((64,), float(d)))                        for d in deltas])
huber_vals = torch.tensor([LC.huber_diff(torch.full((64,), float(d)), huber_delta)        for d in deltas])
charb_vals = torch.tensor([LC.charbonnier_diff(torch.full((64,), float(d)), charb_eps)    for d in deltas])

fig, ax = plt.subplots(figsize=(6.5, 4))
ax.plot(deltas, mse_vals,   label='MSE')
ax.plot(deltas, l1_vals,    label='L1')
ax.plot(deltas, huber_vals, label=f'Huber (delta={huber_delta:g})')
ax.plot(deltas, charb_vals, label=f'Charbonnier (eps={charb_eps:g})')
ax.axvline( huber_delta, color='grey', ls=':', lw=1)
ax.axvline(-huber_delta, color='grey', ls=':', lw=1)
ax.set_xlabel('residual d = pred - target')
ax.set_ylabel('loss value')
ax.set_title('Pointwise curve penalties')
ax.legend(frameon=False)
fig.tight_layout()
plt.show()


## Small-residual regime

Zooming into small residuals shows that Huber and Charbonnier are quadratic
near zero (matching MSE) while L1 stays linear with a sharp corner at the
origin. Charbonnier rounds the L1 corner by `eps`.

In [ ]:
small       = torch.linspace(-0.2, 0.2, 201)
mse_s   = torch.tensor([LC.mse_diff(torch.full((64,), float(d)))                    for d in small])
l1_s    = torch.tensor([LC.l1_diff(torch.full((64,), float(d)))                     for d in small])
huber_s = torch.tensor([LC.huber_diff(torch.full((64,), float(d)), huber_delta)     for d in small])
charb_s = torch.tensor([LC.charbonnier_diff(torch.full((64,), float(d)), charb_eps) for d in small])

fig, ax = plt.subplots(figsize=(6.5, 4))
ax.plot(small, mse_s,   label='MSE')
ax.plot(small, l1_s,    label='L1')
ax.plot(small, huber_s, label='Huber')
ax.plot(small, charb_s, label='Charbonnier')
ax.set_xlabel('residual d')
ax.set_ylabel('loss value')
ax.set_title('Behaviour near zero residual')
ax.legend(frameon=False)
fig.tight_layout()
plt.show()


## Expected visual outcome

MSE is a parabola, L1 a V with constant slope, Huber follows MSE inside
`|d| < delta` then switches to the linear L1 regime at the marked vertical
lines, and Charbonnier tracks L1 for large `|d|` but is smooth at the origin.
All penalties are zero at `d = 0` and symmetric, confirming the elementwise
residual losses behave as designed.